## 1. 問題の分析

### 競技プログラミング視点
- `Array.filter` の内部実装を手動で再現する問題
- 単純な線形走査で O(n) が最適解
- 余分なメモリを最小化するため、結果配列への直接 push が最適

### 業務開発視点
- コールバック関数の型安全性確保（`Fn` 型）
- `fn` の戻り値は `any` → 明示的に `Boolean()` で truthiness を評価
- 入力配列は破壊しない（Pure function）

### TypeScript 特有の考慮点
- `Fn` 型で引数 `(n, i)` の型が保証されており、コンパイル時安全
- `readonly` 修飾子でイミュータブル性を担保可能

---

## 2. アルゴリズムアプローチ比較

| アプローチ | 時間計算量 | 空間計算量 | TS実装コスト | 型安全性 | 可読性 | 備考 |
|---|---|---|---|---|---|---|
| **for ループ + push** | O(n) | O(k) | 低 | 高 | 高 | ✅ 最適解 |
| for...of + entries() | O(n) | O(k) | 低 | 高 | 中 | index 取得にオーバーヘッド |
| reduce | O(n) | O(k) | 中 | 高 | 中 | 関数型スタイルだが若干遅い |

> k = フィルタ後の要素数

---

## 3. 選択したアルゴリズムと理由

- **選択**: `for` ループ + 条件付き `push`
- **理由**:
  - インデックス `i` が直接利用可能で `Fn(n, i)` の呼び出しが最も自然
  - V8エンジンでの JIT 最適化が最も効きやすいシンプルな構造
  - `Boolean()` による明示的な truthiness 評価で型安全性確保
  - 制約 `arr.length <= 1000` の範囲では計算量差は無視できるが、最もシンプルで保守性が高い

---

## 4. 実装コード

```typescript
// Analyze Complexity
// Runtime 42 ms
// Beats 69.47%
// Memory 55.88 MB
// Beats 17.95%
type Fn = (n: number, i: number) => any;

/**
 * 配列をコールバック関数でフィルタリングする（Array.filter の手動実装）
 * @param arr - フィルタ対象の数値配列
 * @param fn  - フィルタ条件コールバック (要素値, インデックス) => truthy/falsy
 * @returns fn が truthy を返した要素のみを含む新しい配列
 * @complexity Time: O(n), Space: O(k)  ※ k = フィルタ後の要素数
 */
function filter(arr: number[], fn: Fn): number[] {
    const result: number[] = [];

    for (let i = 0; i < arr.length; i++) {
        if (Boolean(fn(arr[i], i))) {
            result.push(arr[i]);
        }
    }

    return result;
}
```

---

## ポイント解説

```
arr = [-2, -1, 0, 1, 2],  fn = (n) => n + 1

i=0: fn(-2) = -1  → Boolean(-1) = true  ✅ → result: [-2]
i=1: fn(-1) =  0  → Boolean(0)  = false ❌
i=2: fn( 0) =  1  → Boolean(1)  = true  ✅ → result: [-2, 0]
i=3: fn( 1) =  2  → Boolean(2)  = true  ✅ → result: [-2, 0, 1]
i=4: fn( 2) =  3  → Boolean(3)  = true  ✅ → result: [-2, 0, 1, 2]
```

### TypeScript 特有の最適化ポイント

| 観点 | 本実装での対応 |
|---|---|
| **型安全性** | `Fn` 型でコールバックの引数型を保証 |
| **Truthiness** | `Boolean(fn(...))` で `any` 型の戻り値を安全に評価 |
| **イミュータブル性** | 元の `arr` を破壊せず、新しい `result[]` を返却 |
| **Null安全** | `arr.length` ガードにより空配列は自然に空を返す |

## ボトルネック分析

### Memory 問題: `push` による動的リサイズ

```
push の内部動作（V8エンジン）:
[初期] capacity: 4   → [_, _, _, _]
push×4 → capacity: 4   ✅ 問題なし
push×5 → capacity: 8   ⚠️ 再アロケーション（×2拡張）
push×9 → capacity: 16  ⚠️ 再アロケーション
...
最悪 log₂(n) 回のコピーが発生 → メモリ断片化
```

### Runtime 問題: `Boolean()` のラッパーコスト

```typescript
Boolean(fn(arr[i], i))  // ← 関数呼び出しオーバーヘッド
```

---

## 最適化戦略

```
【Before】動的 push                【After】事前確保 + 書き込みポインタ
result = []                        result = new Array(arr.length)
push → push → push                 result[count++] = value
(内部で何度もリサイズ)               (固定メモリ、リサイズなし)
最終: result                        最終: result.slice(0, count)
```

---

## 最適化実装

```typescript
// Analyze Complexity
// Runtime 38 ms
// Beats 87.22%
// Memory 55.64 MB
// Beats 32.35%
type Fn = (n: number, i: number) => any;

/**
 * 配列をコールバック関数でフィルタリングする（メモリ最適化版）
 * @param arr - フィルタ対象の数値配列
 * @param fn  - フィルタ条件コールバック (要素値, インデックス) => truthy/falsy
 * @returns fn が truthy を返した要素のみを含む新しい配列
 * @complexity Time: O(n), Space: O(n) → 実効 O(k)
 */
function filter(arr: number[], fn: Fn): number[] {
    // ✅ 最大サイズで事前確保 → push による動的リサイズを排除
    const result = new Array<number>(arr.length);
    let count = 0;

    for (let i = 0; i < arr.length; i++) {
        // ✅ Boolean() ラッパー除去 → 暗黙の truthy 評価で直接分岐
        if (fn(arr[i], i)) {
            result[count++] = arr[i];
        }
    }

    // ✅ 実際に使用した分だけ切り出し
    return result.slice(0, count);
}
```

---

## 最適化ポイント早見表

| 項目 | Before | After | 効果 |
|---|---|---|---|
| **メモリ確保** | `[]` + 動的 push | `new Array(n)` 事前確保 | 再アロケーション排除 |
| **Truthiness評価** | `Boolean(fn(...))` | `if (fn(...))` 直接評価 | 関数呼び出しコスト削減 |
| **書き込み方式** | `push()` | `result[count++]` 直接代入 | 配列操作オーバーヘッド削減 |
| **最終出力** | `result` そのまま | `result.slice(0, count)` | 未使用領域を切り捨て |

---

## トレードオフ

```
new Array(n) → slice(0, count) の構造:

メリット: V8 が連続メモリ領域を1回確保 → GC 負荷軽減
デメリット: slice() で最終的に O(k) のコピーが1回発生

→ arr.length <= 1000 の制約下では
  push の累積コピーコスト >> slice の1回コピーコスト
  ∴ トータルでメモリ効率・速度ともに優位
```